# Build data files
Turn the collection of wide snapshot CSV files in `data/` into a single tidy (long) data file: `south-dakota-voter-registration-data.csv`. Also, build a simplified version: `south-dakota-voter-registration-data.csv`. Also, build a JSON file in `_site` to power the chart page. Also, build `README.md` from a basic template.

In [1]:
from pathlib import Path
import json

import pandas as pd

In [2]:
dtype_fips = {
    "state_fips": str,
    "county_fips": str
}

In [3]:
# read in table of election dates
df_elections = pd.read_json("elex-lookup.json", typ="series").to_frame("election_type")
df_elections.index.name = "date"
df_elections.reset_index(inplace=True)
df_elections["date"] = df_elections["date"].astype(str)
df_elections.sort_values("date", ascending=False, inplace=True)

In [4]:
df_elections.head()

,date,election_type
55,2026-06-02,primary
54,2024-11-05,general
53,2024-06-04,primary
52,2022-11-08,general
51,2022-06-07,primary


In [5]:
# read in fips data
df_fips = pd.read_csv(
    "us-county-fips.csv",
    dtype=dtype_fips
)

In [6]:
df_fips.head()

,state_abbr,state_fips,county_fips,county_name,other
0,AL,01,001,Autauga County,H1
1,AL,01,003,Baldwin County,H1
2,AL,01,005,Barbour County,H1
3,AL,01,007,Bibb County,H1
4,AL,01,009,Blount County,H1


In [7]:
# create combined fips code column
df_fips["fips"] = df_fips["state_fips"] + df_fips["county_fips"]

In [8]:
df_fips.head()

,state_abbr,state_fips,county_fips,county_name,other,fips
0,AL,01,001,Autauga County,H1,01001
1,AL,01,003,Baldwin County,H1,01003
2,AL,01,005,Barbour County,H1,01005
3,AL,01,007,Bibb County,H1,01007
4,AL,01,009,Blount County,H1,01009


In [9]:
# filter to get just SD records (b/c we're joining on county names,
# which are not unique among states)
df_sd_fips = df_fips[df_fips["state_fips"] == "46"]

In [10]:
df_sd_fips.head()

,state_abbr,state_fips,county_fips,county_name,other,fips
2362,SD,46,003,Aurora County,H1,46003
2363,SD,46,005,Beadle County,H1,46005
2364,SD,46,007,Bennett County,H1,46007
2365,SD,46,009,Bon Homme County,H1,46009
2366,SD,46,011,Brookings County,H1,46011


In [11]:
# test: 66 counties in SD
assert len(df_sd_fips) == 66

In [12]:
# clean up county name for joining later
df_sd_fips["county_name"] = df_sd_fips["county_name"].str.replace(" County", "")

In [13]:
df_sd_fips.head()

,state_abbr,state_fips,county_fips,county_name,other,fips
2362,SD,46,003,Aurora,H1,46003
2363,SD,46,005,Beadle,H1,46005
2364,SD,46,007,Bennett,H1,46007
2365,SD,46,009,Bon Homme,H1,46009
2366,SD,46,011,Brookings,H1,46011


In [14]:
# add two rows to supplement the data
new_rows = pd.DataFrame([
    {
        "county_name": "Oglala Lakota",
        "fips": "46102",
        "state_abbr": "SD",
        "state_fips": "46",
        "county_fips": "102",
        "other": "H1"
    },
    {
        "county_name": "Washabaugh",
        "fips": "46131",
        "state_abbr": "SD",
        "state_fips": "46",
        "county_fips": "131",
        "other": "H1"
    }
])

df_sd_fips = pd.concat([df_sd_fips, new_rows])

In [15]:
df_sd_fips.tail()

,state_abbr,state_fips,county_fips,county_name,other,fips
2425,SD,46,129,Walworth,H1,46129
2426,SD,46,135,Yankton,H1,46135
2427,SD,46,137,Ziebach,H1,46137
0,SD,46,102,Oglala Lakota,H1,46102
1,SD,46,131,Washabaugh,H1,46131


In [16]:
# read all individual data files into one dataframe
data_files = Path("data").glob("*.csv")
df_data = pd.concat([pd.read_csv(x, dtype=dtype_fips) for x in data_files])
df_data.sort_values("date", ascending=False, inplace=True)
df_data["date"] = df_data["date"].astype(str)

In [17]:
df_data.head()

,date,county,libertarian,republican,democratic,npa_ind,other,inactive,independent,npa,npa_ind_oth,no_labels,constitution,americans_elect,reform
3,2026-09-01,Bon Homme,10.0,2473,781,NaN,NaN,249.0,463.0,232.0,NaN,NaN,NaN,NaN,NaN
27,2026-09-01,Hamlin,8.0,3135,499,NaN,1.0,208.0,359.0,294.0,NaN,NaN,NaN,NaN,NaN
21,2026-09-01,Edmunds,4.0,1868,420,NaN,2.0,131.0,229.0,172.0,NaN,NaN,NaN,NaN,NaN
22,2026-09-01,Fall River,31.0,4003,758,NaN,3.0,897.0,784.0,648.0,NaN,NaN,NaN,NaN,NaN
23,2026-09-01,Faulk,NaN,1159,147,NaN,NaN,84.0,194.0,39.0,NaN,NaN,NaN,NaN,NaN


In [18]:
# melt dataframe wide to long
id_cols = ["date", "county"]
value_cols = [x for x in df_data.columns if x not in id_cols]

df_melted = df_data.melt(
    id_vars=id_cols,
    var_name="party",
    value_vars=value_cols,
    value_name="voters"
)

In [19]:
df_melted.head()

,date,county,party,voters
0,2026-09-01,Bon Homme,libertarian,10.0
1,2026-09-01,Hamlin,libertarian,8.0
2,2026-09-01,Edmunds,libertarian,4.0
3,2026-09-01,Fall River,libertarian,31.0
4,2026-09-01,Faulk,libertarian,NaN


In [20]:
# remove rows with null/0 voters
df_no_nulls = df_melted[
    (df_melted["voters"].notna() &
     df_melted["voters"] > 0)
]

In [21]:
# sort by date, county, voters
df_no_nulls.sort_values(
    ["date", "county", "voters"],
    ascending=[True, True, False],
    inplace=True
)

In [22]:
df_no_nulls.head()

,date,county,party,voters
36983,1976-05-17,Aurora,democratic,1508.0
24639,1976-05-17,Aurora,republican,984.0
61671,1976-05-17,Aurora,other,190.0
37031,1976-05-17,Beadle,democratic,7007.0
24687,1976-05-17,Beadle,republican,4773.0


In [23]:
# join in election info
df_final = df_no_nulls.merge(
    df_elections,
    on="date",
    how="left"
# join in fips codes
).merge(
    df_sd_fips,
    left_on="county",
    right_on="county_name",
    how="left"
# select just the columns we want and fill nulls with empty strings
)[[
    "date",
    "county",
    "party",
    "voters",
    "election_type",
    "fips"
]].fillna("")

In [24]:
df_final.head()

,date,county,party,voters,election_type,fips
0,1976-05-17,Aurora,democratic,1508.0,primary,46003
1,1976-05-17,Aurora,republican,984.0,primary,46003
2,1976-05-17,Aurora,other,190.0,primary,46003
3,1976-05-17,Beadle,democratic,7007.0,primary,46005
4,1976-05-17,Beadle,republican,4773.0,primary,46005


In [25]:
# write to file
df_final.to_csv(
    "south-dakota-voter-registration-data.csv",
    index=False,
    float_format="%.0f"
)

In [26]:
"""
make a simplified version:
- remove inactive voter records
- collapse non R/D parties into "other"
- remove two snapshot records for Washabaugh County, which merged with Jackson County in 1983
- merge records of Shannon County, which was renamed Oglala Lakota County in 2015
"""

# remove inactive
df_final = df_final[df_final["party"] != "inactive"]

# collapse non R/D parties into an "other" category
df_final["party"] = df_final["party"].apply(
    lambda x: x if x in ["republican", "democratic"] else "other"
)

# ... and sum the groups of "other" rows
# get current column order
df_cols = df_final.columns
df_final = df_final.groupby(
    [x for x in df_final.columns if x != "voters"],
    dropna=False,
    as_index=False
)['voters'].sum()[df_cols]

# remove Washabaugh records
df_final = df_final[df_final["county"] != "Washabaugh"]

# rename Shannon County records
df_final["county"] = df_final["county"].apply(
    lambda x: "Oglala Lakota" if x == "Shannon" else x
)

# ... and update the fips
ol_fips = new_rows[new_rows["county_name"] == "Oglala Lakota"]["fips"].iloc[0]
df_final["fips"] = df_final["fips"].apply(
    lambda x: ol_fips if x == "46113" else x
)

In [27]:
df_final.head()

,date,county,party,voters,election_type,fips
0,1976-05-17,Aurora,democratic,1508.0,primary,46003
1,1976-05-17,Aurora,other,190.0,primary,46003
2,1976-05-17,Aurora,republican,984.0,primary,46003
3,1976-05-17,Beadle,democratic,7007.0,primary,46005
4,1976-05-17,Beadle,other,1264.0,primary,46005


In [28]:
# write to file
df_final.to_csv(
    "south-dakota-voter-registration-data-simplified.csv",
    index=False,
    float_format="%.0f"
)

In [29]:
df_final.groupby("party")["voters"].sum().reset_index().sort_values("voters")

,party,voters
1,other,22379478.0
0,democratic,30173043.0
2,republican,49392168.0


In [ ]:
# next, build a JSON file to match the format expected by
# a chart.js area chart - includes a statewide series plus
# a per-county breakdown (keyed by fips) so index.html only
# ever has to fetch this one file

df_final["date"] = pd.to_datetime(df_final["date"])
df_final = df_final.sort_values("date")

party_colors = {
    "democratic": {
        "border": "#1E40AF",
        "fill": "rgba(30, 64, 175, 0.65)"
    },
    "republican": {
        "border": "#B91C1C",
        "fill": "rgba(185, 28, 28, 0.65)"
    },
    "other": {
        "border": "#B45309",
        "fill": "rgba(180, 83, 9, 0.65)"
    }
}

parties = ["democratic", "republican", "other"]


def build_datasets(df, labels):
    datasets = []

    for party in parties:
        party_df = (
            df[df["party"] == party]
            .groupby("date", as_index=False)["voters"]
            .sum()
            .sort_values("date")
        )

        series = (
            party_df.set_index("date")["voters"]
            .reindex(labels, fill_value=0)
            .tolist()
        )

        colors = party_colors.get(party, {
            "border": "#64748b",
            "fill": "rgba(100, 116, 139, 0.25)"
        })

        datasets.append({
            "label": party.title(),
            "data": series,
            "fill": True,
            "borderColor": colors["border"],
            "backgroundColor": colors["fill"],
            "tension": 0.3
        })

    return datasets


statewide_labels = sorted(df_final["date"].unique())

chart_data = {
    "labels": [d.strftime("%Y-%m-%d") for d in statewide_labels],
    "datasets": build_datasets(df_final, statewide_labels),
    "counties": {}
}

for fips, county_df in df_final.groupby("fips"):
    county_name = county_df["county"].iloc[0]
    county_labels = sorted(county_df["date"].unique())

    chart_data["counties"][fips] = {
        "name": county_name,
        "labels": [d.strftime("%Y-%m-%d") for d in county_labels],
        "datasets": build_datasets(county_df, county_labels)
    }

with open("docs/south-dakota-voter-registration.json", "w") as outfile:
    json.dump(chart_data, outfile)

# Build README.md
Fill in the `{% updated %}` and `{% snapshot_count %}` placeholders in `README-template.md` and write the result to `README.md`.

In [31]:
from datetime import datetime, timezone

with open("README-template.md", "r") as infile:
    readme_content = infile.read()

updated = datetime.now(timezone.utc).date().strftime("%B %-d, %Y")
snapshot_count = len(list(Path("data").glob("*.csv")))

readme_content = readme_content.replace(
    "{% updated %}", updated
).replace(
    "{% snapshot_count %}", str(snapshot_count)
)

with open("README.md", "w") as outfile:
    outfile.write(readme_content)

print(f"Wrote README.md ({snapshot_count} snapshots, updated {updated})")

Wrote README.md (187 snapshots, updated September 12, 2026)
